# 0.19 — Entity-centric theme emergence (novelty + bridging + clustering, tracked)

New detection object: a **novel entity** (or new entity relationship) that **bridges disconnected
contexts** and whose reach **grows** over weeks. This supersedes 0.18's novel-*term* census, which
conflated entities (`chatgpt`) with events (`ftx collapse`, `ohio train`) and let persistent events
dominate the promoted shortlist.

**Pipeline (each stage discards more; fully unsupervised — no theme keywords):**

| Stage | Test | Kills |
|---|---|---|
| 0 **Entity selection** | term is not an event/transaction predicate | `ftx collapse`, `adani rout`, `turkey quake` |
| 1 **Novelty** | entity baseline-unseen (≤2022-09-30) | `microsoft`, `opec`, `bankruptcy` |
| 2 **Bridging + clustering** | many partners, partners *don't* co-occur (low clustering = hub, not clique) | `ftx`/`binance`/`alameda` clique; single-IPO entities |
| 3 **Track + promote** | alive ≥ N weeks AND reach at new high | one-off spikes |
| 4 **Subgraph assembly** | anchor + its entity neighbors = a multi-actor cluster | (repairs assembly) |
| 5 **LLM reject-confirm** | generic, entity-free taxonomy, diverse evidence | residual non-themes |

**Benchmark:** generative AI. ChatGPT 2022-11-30 · CHAT ETF 2023-05-17. Target: catch the genAI
*entity* in early December, demote the FTX/Adani events via **clustering**, and hand the LLM a genuinely
multi-actor genAI cluster.

*Stage 0 uses a dependency-free generic event-predicate list (spaCy NER is the documented upgrade).*

In [ ]:
import os, re
from collections import Counter, defaultdict
from itertools import combinations
from pathlib import Path

import networkx as nx
import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS

_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
OUTPUT_DIR = _ROOT / "notebooks" / "output"
ENV_PATH = _ROOT / ".env"
if ENV_PATH.exists():
    for _l in ENV_PATH.read_text().splitlines():
        _l = _l.strip()
        if "=" in _l and not _l.startswith("#"):
            _k, _, _v = _l.partition("=")
            os.environ.setdefault(_k.strip(), _v.strip().strip('"').strip("'"))

BASELINE_END = pd.Timestamp("2022-09-30")
DISCOVERY_START = pd.Timestamp("2022-10-01")
DISCOVERY_END = pd.Timestamp("2023-02-28")
CHATGPT_LAUNCH = pd.Timestamp("2022-11-30")
INCEPTION = pd.Timestamp("2023-05-17")

FREQ = "W-MON"; REP_HL = 8
MIN_MENTIONS = 3        # candidate must appear >= this many times in the week
DEGREE_MIN = 8          # catch: entity must reach >= this many distinct entity partners (high recall)
PERSIST_WEEKS = 2       # promote: alive >= this many weeks AND reach at a new high
CLUSTER_K = 15          # clustering coefficient computed over the top-K entity partners
CLUSTER_MAX = 0.60      # promote only bridges (low clustering); cliques (events) are demoted
LLM_MAX_GROUPS = 25

TERM_STOP = set(ENGLISH_STOP_WORDS) | {"says", "said", "new", "year", "week", "report", "shares", "stock",
    "jan", "feb", "mar", "apr", "may", "jun", "jul", "aug", "sep", "oct", "nov", "dec"}
# Stage 0: generic event / transaction predicates (NOT theme-specific — like stopwords)
EVENT_STOP = {
    "collapse", "rout", "bankruptcy", "bankrupt", "fraud", "lawsuit", "sue", "sues", "sued", "probe", "hearing",
    "trial", "court", "arrest", "arrested", "resign", "resigns", "ban", "bans", "banned", "outage", "recall",
    "default", "slump", "slumps", "plunge", "plunges", "crash", "derailment", "strike", "quake", "earthquake",
    "protest", "protests", "unrest", "attack", "war", "sanctions", "fine", "fined", "scandal", "layoffs",
    "layoff", "pivot", "halt", "halts", "delay", "delays", "death", "dies", "killed", "guilty", "charges",
    "charged", "indicted", "crisis", "takeover", "merger", "deal", "acquires", "acquire", "buys", "stake",
    "ipo", "listing", "bond", "bonds", "notes", "debt", "offering", "case", "settlement", "shortage",
    "shutdown", "tumbles", "soars", "jumps", "rises", "falls", "drops", "gains", "cuts", "raises",
}
HINT = re.compile(r"chatgpt|openai|chatbot|generative|\bgpt-", re.I)   # validation only

def is_entity(t): return not any(tok in EVENT_STOP for tok in t.split())
def normalize_terms(v): return v.tolist() if isinstance(v, np.ndarray) else (list(v) if isinstance(v, (list, tuple)) else [])
def filter_terms(ts): return [t for t in ts if len(t) >= 3 and not any(tok in TERM_STOP for tok in t.split())]
def week_ts(w): return pd.Period(w, freq=FREQ).start_time
def clustcoef(P, adj):
    if len(P) < 2: return 0.0
    tot = pairs = 0
    for i in range(len(P)):
        for j in range(i + 1, len(P)):
            tot += 1
            if P[j] in adj[P[i]]: pairs += 1
    return pairs / tot if tot else 0.0
print("setup ok ·", f"DEGREE_MIN={DEGREE_MIN} CLUSTER_MAX={CLUSTER_MAX} · {len(EVENT_STOP)} event-stop words")

In [ ]:
def detect(news, baseline_end, ds, de):
    """One row per (novel ENTITY, week): bridging degree, clustering coefficient, subgraph, headlines."""
    df = news.copy()
    df["terms_f"] = df["terms"].map(normalize_terms).map(filter_terms)
    df["week"] = df["date"].dt.to_period(FREQ).astype(str)
    baseline_terms = set()
    for tl in df.loc[df.date <= baseline_end, "terms_f"]:
        baseline_terms.update(tl)
    disc = [w for w in sorted(df["week"].unique(), key=week_ts) if ds <= week_ts(w) <= de]

    rows = []
    for week in disc:
        grp = df.loc[df.week == week]
        twc, adj = Counter(), defaultdict(Counter)
        for tl in grp["terms_f"]:
            s = set(tl)
            for t in s:
                twc[t] += 1
            for a, b in combinations(sorted(s), 2):
                adj[a][b] += 1; adj[b][a] += 1
        for t, cnt in twc.items():
            if t in baseline_terms or cnt < MIN_MENTIONS:        # §1 novelty + support
                continue
            if not is_entity(t):                                 # §0 entity selection
                continue
            epart = [p for p, _ in adj[t].most_common() if is_entity(p)]   # §2 entity partners
            P = epart[:CLUSTER_K]
            reps = [hl for hl, tl in zip(grp["Headline"], grp["terms_f"]) if t in set(tl)][:REP_HL]
            rows.append({"week": week, "anchor": t, "mentions": int(cnt),
                         "anchor_degree": len(epart), "clustering": round(clustcoef(P, adj), 3),
                         "subgraph": [t] + P[:10], "rep_headlines": reps})
    return pd.DataFrame(rows)

In [ ]:
# --- load and run the entity-novelty census ---
meta = pd.read_parquet(OUTPUT_DIR / "genai_full_meta.parquet")
terms = pd.read_parquet(OUTPUT_DIR / "genai_graph_terms.parquet")
meta["date"] = pd.to_datetime(meta["date"]).dt.normalize()
terms["date"] = pd.to_datetime(terms["date"]).dt.normalize()
terms = terms.drop_duplicates(["Headline", "date"], keep="first")
news = meta.merge(terms[["Headline", "date", "terms"]], on=["Headline", "date"], how="left")
news = news.loc[news.date <= DISCOVERY_END]

R = detect(news, BASELINE_END, DISCOVERY_START, DISCOVERY_END)
R["genai"] = R["anchor"].str.contains(HINT, na=False)
print(f"{len(R):,} (novel entity x week) rows · {R.anchor.nunique():,} distinct novel entities")
print("\ngenAI entities — weekly mentions / degree / clustering:")
print(R.loc[R.genai, ["week", "anchor", "mentions", "anchor_degree", "clustering"]]
      .sort_values(["week", "anchor"]).to_string(index=False))

In [ ]:
# ---------- §3 high-recall CATCH ----------
R["caught"] = (R.mentions >= MIN_MENTIONS) & (R.anchor_degree >= DEGREE_MIN)
pw = R[R.caught].groupby("week").anchor.nunique()
print(f"caught (entity x week): {int(R.caught.sum()):,} · distinct caught entities: {R[R.caught].anchor.nunique():,}"
      f" · per week median {int(pw.median())}")
print("genAI entities caught:", sorted(R[R.genai & R.caught].anchor.unique())[:15])

In [ ]:
# ---------- §2/§3 promote: persist + growth + LOW clustering (bridge, not clique) ----------
def promote(R):
    out = []
    for a, sub in R[R.caught].groupby("anchor"):
        sub = sub.sort_values("week", key=lambda s: s.map(week_ts))
        wk, deg, clu = list(sub.week), list(sub.anchor_degree), list(sub.clustering)
        p = None
        for i in range(len(wk)):
            if (i + 1) >= PERSIST_WEEKS and deg[i] >= max(deg[:i] or [0]) and float(np.median(clu[:i + 1])) <= CLUSTER_MAX:
                p = wk[i]; break
        out.append({"anchor": a, "first_caught": wk[0], "n_weeks": len(wk), "deg_max": max(deg),
                    "med_clustering": round(float(np.median(clu)), 3), "promoted_week": p,
                    "genai": bool(HINT.search(a))})
    return pd.DataFrame(out)

P = promote(R)
promoted = P[P.promoted_week.notna()].copy()
print(f"promoted: {len(promoted)} of {len(P)} caught entities")
print("\ngenAI entity lifecycle:")
print(P[P.genai].sort_values("first_caught")[
      ["anchor", "first_caught", "n_weeks", "deg_max", "med_clustering", "promoted_week"]].head(12).to_string(index=False))
print("\nclustering check — does it separate themes from event-cliques?")
print("  genAI med clustering   :", sorted(P[P.genai].med_clustering.tolist())[:8])
ev = P[P.anchor.str.contains("ftx|adani|bankman|binance", case=False, na=False)]
print("  FTX/Adani med clustering:", ev[["anchor", "med_clustering", "promoted_week"]].to_string(index=False) if len(ev) else "(none caught)")

In [ ]:
# ---------- §4 subgraph assembly + §5 unsupervised LLM confirm (diverse evidence) ----------
from typing import Literal
from pydantic import BaseModel

pset = set(promoted.anchor)
G = nx.Graph(); G.add_nodes_from(pset)
for _, r in R[R.anchor.isin(pset) & R.caught].iterrows():        # merge co-occurring promoted entities
    for p in r.subgraph:
        if p in pset and p != r.anchor:
            G.add_edge(r.anchor, p)
groups = [sorted(c) for c in nx.connected_components(G)]

def diverse_evidence(anchors):
    pairs = []
    for _, r in R[R.anchor.isin(anchors) & R.caught].sort_values("week", key=lambda s: s.map(week_ts)).iterrows():
        for hl in r.rep_headlines:
            pairs.append(hl)
    seen, out = set(), []
    for hl in pairs:                                            # dedupe, keep order (early..late = context spread)
        k = hl.lower()
        if k not in seen:
            seen.add(k); out.append(hl)
    return out[::max(1, len(out) // 10)][:10] if len(out) > 10 else out

def members_subgraph(anchors):
    s = set()
    for sg in R[R.anchor.isin(anchors) & R.caught].subgraph:
        s.update(sg)
    return sorted(s)

gdf = pd.DataFrame({"anchors": groups})
gdf["reach"] = gdf.anchors.map(lambda a: int(P.set_index("anchor").loc[a, "deg_max"].max()))
gdf["genai"] = gdf.anchors.map(lambda a: any(HINT.search(x) for x in a))
gdf["subgraph"] = gdf.anchors.map(members_subgraph)
gdf = gdf.sort_values("reach", ascending=False).reset_index(drop=True)

REJECT_SYS = (
    "You are a conservative FILTER that REMOVES clusters of news headlines that are NOT emerging themes. "
    "You never decide what IS a theme; you only flag clusters that CLEARLY fall into one of three non-theme "
    "categories, judging ONLY from the headlines shown. If a cluster does not clearly match a reject category, "
    "you MUST keep it. When in doubt, KEEP.\n\n"
    "1. single_entity_event - ONE company/person's idiosyncratic event with no broader multi-actor narrative.\n"
    "2. macro_market_aggregate - generic market/macro conditions: rates, inflation, FX, yields, indices, "
    "central-bank policy, commodities, GDP.\n"
    "3. boilerplate_wire - wire formatting, calendars, generic 'shares rise/fall', routine corporate PR.\n\n"
    "KEEP anything describing a SPECIFIC technological, industrial, product, or policy development connecting "
    "multiple actors - even if reported via a deal or an event."
)
class Reject(BaseModel):
    verdict: Literal["keep", "reject"]
    category: Literal["single_entity_event", "macro_market_aggregate", "boilerplate_wire", "none"]
    reason: str
_client = None
def judge(subgraph, headlines):
    global _client
    from openai import OpenAI
    if _client is None:
        _client = OpenAI(api_key=os.environ["OPENAI_API_KEY"], base_url=os.environ.get("OPENAI_BASE") or None)
    user = ("Cluster entities: " + ", ".join(subgraph[:14]) + "\nRepresentative headlines:\n"
            + "\n".join(f"- {h}" for h in headlines) + "\n\nClassify this cluster.")
    r = _client.beta.chat.completions.parse(
        model=os.environ.get("OPENAI_DEFAULT_MODEL", "gpt-4o-mini"), temperature=0, response_format=Reject,
        messages=[{"role": "system", "content": REJECT_SYS}, {"role": "user", "content": user}])
    p = r.choices[0].message.parsed
    return (p.verdict == "keep"), p.category, p.reason

gdf["llm_keep"], gdf["llm_cat"] = None, None
for i, g in gdf.head(LLM_MAX_GROUPS).iterrows():
    keep, cat, _ = judge(g.subgraph, diverse_evidence(g.anchors))
    gdf.at[i, "llm_keep"], gdf.at[i, "llm_cat"] = keep, cat
print(f"assembled {len(groups)} promoted themes · LLM-confirmed top {min(LLM_MAX_GROUPS, len(gdf))} "
      f"(kept {int((gdf.llm_keep == True).sum())})")

In [ ]:
# ---------- RESULT vs benchmark ----------
gen_caught = R[R.genai & R.caught]
gen_first = gen_caught.week.min() if len(gen_caught) else None
gp = P[P.genai & P.promoted_week.notna()]
gen_prom = gp.promoted_week.min() if len(gp) else None
gai = gdf[gdf.genai]

print("=" * 72)
print("genAI benchmark (no keywords at detection):")
print(f"  first CAUGHT (entity) : {gen_first}")
print(f"  PROMOTED (confirmed)  : {gen_prom}")
print(f"  ChatGPT 2022-11-30 · CHAT ETF 2023-05-17")
if gen_prom:
    lead = (INCEPTION - week_ts(gen_prom)).days
    print(f"  lead vs CHAT ETF      : {lead} days (~{lead//30} months)")
if len(gai):
    g0 = gai.iloc[0]
    print(f"  genAI subgraph        : {', '.join(g0.subgraph[:12])}")
    print(f"  LLM verdict           : {'KEEP' if g0.llm_keep else 'reject/' + str(g0.llm_cat)}")
print("=" * 72)
print(f"funnel: {R.anchor.nunique():,} novel entities -> {R[R.caught].anchor.nunique():,} caught "
      f"-> {len(promoted)} promoted -> {len(groups)} themes -> {int((gdf.llm_keep == True).sum())} LLM-kept")
print("\nLLM-kept promoted themes (shortlist a human would inspect):")
for _, g in gdf[gdf.llm_keep == True].head(20).iterrows():
    print(f"  reach {int(g.reach):4}{' ★genAI' if g.genai else '       '}  {', '.join(g.subgraph[:8])}")

R.to_parquet(OUTPUT_DIR / "genai_entity_anchor_weeks.parquet", index=False)
P.to_parquet(OUTPUT_DIR / "genai_entity_promotion.parquet", index=False)
print("\nsaved -> genai_entity_anchor_weeks.parquet · genai_entity_promotion.parquet")